# ViSoBERT + CNN v2 — train and test CONCAT + GATED in one run

Attach the saved output of `00_prepare_vihsd.ipynb`, select **GPU T4 x2**, enable Internet, add `HF_TOKEN`, and use **Run All**. The notebook runs two isolated experiments sequentially: `concat` first, then `gated`. Each experiment initializes a fresh ViSoBERT, trains, restores its best validation checkpoint, calibrates class biases on validation only, evaluates test, and saves artifacts in its own subdirectory.

In [ ]:
# ===== USER SETTINGS =====
MODEL_VARIANTS = ('concat', 'gated')  # both run sequentially in this notebook
SEED = 42
EPOCHS = 10.0          # main run: ~6020 optimizer steps
EVALUATE_TEST = True   # train first, then test the selected model automatically
MICRO_BATCH = 8        # if OOM: 4
GRAD_ACCUMULATION = 5  # if MICRO_BATCH=4: use 10
WEIGHT_DECAY = 0.01
CNN_KERNELS = (2, 3, 4)
CNN_CHANNELS = 64
FUSION_DIM = 256
HEAD_DROPOUT = 0.15
ENCODER_LR = 1.5e-5
HEAD_LR = 5e-5
CLASS_WEIGHT_POWER = 0.5  # sqrt inverse frequency; softer than full inverse
FOCAL_GAMMA = 1.5

# Must run before importing torch or transformers.
import os, sys, subprocess
from datetime import datetime, timezone
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
if MODEL_VARIANTS != ('concat', 'gated'):
    raise ValueError("NB02 must compare exactly ('concat', 'gated')")
assert EVALUATE_TEST is True, 'NB02 must test the selected model after training.'
assert MICRO_BATCH * GRAD_ACCUMULATION == 40
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'datasets', 'accelerate', 'sentencepiece',
    'protobuf', 'huggingface_hub', 'scikit-learn', 'pandas', 'pyarrow',
], check=True)
RUN_TAG = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
ROOT_RUN_DIR = Path('/kaggle/working') / f'proposed_concat_vs_gated_seed{SEED}_{RUN_TAG}'
ROOT_RUN_DIR.mkdir(parents=True, exist_ok=False)
print({
    'model_variants': MODEL_VARIANTS, 'root_run_dir': str(ROOT_RUN_DIR),
    'effective_batch': MICRO_BATCH * GRAD_ACCUMULATION,
})

In [ ]:
import hashlib, json
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
from transformers import AutoTokenizer, set_seed

assert torch.cuda.is_available(), 'Enable GPU T4 x2 in Kaggle Settings.'
assert torch.cuda.device_count() == 1, 'Restart and Run All from the first cell.'
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
set_seed(SEED)
try:
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    HF_ACCOUNT = HfApi(token=HF_TOKEN).whoami()['name']
except Exception as exc:
    raise RuntimeError('Enable a valid Kaggle Secret named HF_TOKEN.') from exc
manifest_paths = list(Path('/kaggle/input').rglob('vihsd_manifest.json'))
if len(manifest_paths) != 1:
    raise RuntimeError(f'Attach exactly one prepared ViHSD output; found {manifest_paths}')
manifest_path = manifest_paths[0]
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['artifact'] == 'vihsd_prepared_v1'
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
frames = {}
for split, spec in manifest['splits'].items():
    path = manifest_path.parent / spec['file']
    assert sha256(path) == spec['sha256']
    frame = pd.read_parquet(path)
    assert len(frame) == spec['rows']
    frames[split] = frame
MODEL_ID, MODEL_SHA = manifest['model_id'], manifest['model_revision']
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_SHA, token=HF_TOKEN
)
def make_dataset(frame):
    dataset = Dataset.from_dict({
        'text': frame['text'].fillna('').astype(str).tolist(),
        'labels': frame['labels'].astype(int).tolist(),
    })
    return dataset.map(
        lambda batch: tokenizer(
            batch['text'], truncation=True, max_length=128, padding=False
        ), batched=True, remove_columns=['text'],
    )
train_ds = make_dataset(frames['train'])
validation_ds = make_dataset(frames['validation'])
train_counts = np.bincount(frames['train']['labels'].astype(int), minlength=3)
if len(train_counts) != 3 or np.any(train_counts == 0):
    raise ValueError(f'Every class must occur in train: {train_counts.tolist()}')
class_weights = train_counts.astype(np.float64) ** (-CLASS_WEIGHT_POWER)
class_weights = class_weights / class_weights.mean()
print({
    'hf_account': HF_ACCOUNT, 'gpu': torch.cuda.get_device_name(0),
    'model_revision': MODEL_SHA, 'dataset_revision': manifest['dataset_revision'],
    'rows': {split: len(frame) for split, frame in frames.items()},
    'train_class_counts': train_counts.tolist(),
    'class_weights': class_weights.round(4).tolist(),
})

In [ ]:
from torch import nn
import torch.nn.functional as F
from transformers import AutoConfig, AutoModel, PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import SequenceClassifierOutput

LABEL2ID = {'CLEAN': 0, 'OFFENSIVE': 1, 'HATE': 2}
ID2LABEL = {value: key for key, value in LABEL2ID.items()}

class ProposedConfig(PretrainedConfig):
    model_type = 'visobert-cnn-v2'
    is_composition = True
    def __init__(self, encoder_config=None, encoder_model_id='uitnlp/visobert',
                 encoder_revision=None, variant='gated',
                 cnn_kernels=(2, 3, 4), cnn_channels=64,
                 fusion_dim=256, head_dropout=0.1, class_weights=None,
                 focal_gamma=1.5, num_labels=3, **kwargs):
        if variant not in ('cnn', 'concat', 'gated'):
            raise ValueError(f'Unsupported variant: {variant}')
        super().__init__(num_labels=num_labels, **kwargs)
        self.encoder_config = dict(encoder_config or {})
        self.encoder_model_id = encoder_model_id
        self.encoder_revision = encoder_revision
        self.variant = variant
        self.cnn_kernels = list(cnn_kernels)
        self.cnn_channels = int(cnn_channels)
        self.fusion_dim = int(fusion_dim)
        self.head_dropout = float(head_dropout)
        self.class_weights = list(class_weights or [])
        self.focal_gamma = float(focal_gamma)

class MaskedCnnPool(nn.Module):
    def __init__(self, hidden_size, channels, kernels):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv1d(hidden_size, channels, kernel_size=kernel) for kernel in kernels
        ])
    @property
    def output_size(self):
        return 2 * sum(conv.out_channels for conv in self.convs)
    def forward(self, hidden_states, attention_mask):
        mask = attention_mask.bool()
        x = (hidden_states * mask.unsqueeze(-1).to(hidden_states.dtype)).transpose(1, 2)
        pooled = []
        for conv in self.convs:
            kernel = conv.kernel_size[0]
            left, right = (kernel - 1) // 2, kernel - 1 - (kernel - 1) // 2
            features = F.gelu(conv(F.pad(x, (left, right))))
            features = features.masked_fill(
                ~mask[:, None, :], torch.finfo(features.dtype).min
            )
            maximum = features.amax(dim=-1)
            valid = mask[:, None, :].to(features.dtype)
            mean = features.masked_fill(~mask[:, None, :], 0).sum(-1) / valid.sum(-1).clamp_min(1)
            pooled.extend((maximum, mean))
        return torch.cat(pooled, dim=-1)

class ProposedViSoBERT(PreTrainedModel):
    config_class = ProposedConfig
    base_model_prefix = 'encoder'
    main_input_name = 'input_ids'
    accepts_loss_kwargs = False
    def __init__(self, config):
        super().__init__(config)
        encoder_payload = dict(config.encoder_config)
        model_type = encoder_payload.pop('model_type')
        encoder_config = AutoConfig.for_model(model_type, **encoder_payload)
        self.encoder = AutoModel.from_config(encoder_config, add_pooling_layer=False)
        hidden = int(encoder_config.hidden_size)
        self.local_pool = MaskedCnnPool(hidden, config.cnn_channels, config.cnn_kernels)
        self.local_projection = nn.Linear(self.local_pool.output_size, config.fusion_dim)
        self.local_norm = nn.LayerNorm(config.fusion_dim)
        if config.variant == 'cnn':
            classifier_input = config.fusion_dim
        else:
            self.global_projection = nn.Linear(hidden, config.fusion_dim)
            self.global_norm = nn.LayerNorm(config.fusion_dim)
            if config.variant == 'concat':
                classifier_input = 2 * config.fusion_dim
            else:
                self.gate = nn.Linear(2 * config.fusion_dim, config.fusion_dim)
                self.fusion_norm = nn.LayerNorm(config.fusion_dim)
                classifier_input = config.fusion_dim
        self.dropout = nn.Dropout(config.head_dropout)
        self.classifier = nn.Linear(classifier_input, config.num_labels)
        self.register_buffer('class_weight_tensor', torch.tensor(
            config.class_weights, dtype=torch.float32
        ), persistent=False)
        self.post_init()
    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        if attention_mask is None:
            raise ValueError('attention_mask is required')
        kwargs.pop('num_items_in_batch', None)  # Trainer API; never forward to encoder
        outputs = self.encoder(
            input_ids=input_ids, attention_mask=attention_mask, return_dict=True, **kwargs
        )
        local_vector = self.local_norm(F.gelu(self.local_projection(
            self.local_pool(outputs.last_hidden_state, attention_mask)
        )))
        if self.config.variant == 'cnn':
            fused = local_vector
        else:
            mask = attention_mask.unsqueeze(-1).to(outputs.last_hidden_state.dtype)
            global_mean = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1).clamp_min(1)
            global_vector = self.global_norm(F.gelu(self.global_projection(global_mean)))
            if self.config.variant == 'concat':
                fused = torch.cat([global_vector, local_vector], dim=-1)
            else:
                alpha = torch.sigmoid(self.gate(
                    torch.cat([global_vector, local_vector], dim=-1)
                ))
                fused = self.fusion_norm(
                    alpha * global_vector + (1.0 - alpha) * local_vector
                    + 0.5 * (global_vector + local_vector)
                )
        logits = self.classifier(self.dropout(fused))
        loss = None
        if labels is not None:
            per_sample = F.cross_entropy(
                logits, labels, weight=self.class_weight_tensor.to(logits.dtype), reduction='none'
            )
            p_true = logits.softmax(-1).gather(1, labels[:, None]).squeeze(1)
            loss = (((1.0 - p_true) ** self.config.focal_gamma) * per_sample).mean()
        return SequenceClassifierOutput(loss=loss, logits=logits)

ARCHITECTURE_NAMES = {
    'cnn': 'ViSoBERT + masked max/mean CNN v2',
    'concat': 'ViSoBERT Global-Local Concat CNN v2',
    'gated': 'ViSoBERT Global-Local Gated CNN v2',
}
encoder_config = AutoConfig.from_pretrained(MODEL_ID, revision=MODEL_SHA, token=HF_TOKEN)
def build_fresh_model(variant):
    proposed_config = ProposedConfig(
        encoder_config=encoder_config.to_dict(), encoder_model_id=MODEL_ID,
        encoder_revision=MODEL_SHA, variant=variant, cnn_kernels=CNN_KERNELS,
        cnn_channels=CNN_CHANNELS, fusion_dim=FUSION_DIM,
        head_dropout=HEAD_DROPOUT, class_weights=class_weights.tolist(),
        focal_gamma=FOCAL_GAMMA, num_labels=3,
        label2id=LABEL2ID, id2label=ID2LABEL,
    )
    fresh_model = ProposedViSoBERT(proposed_config)
    pretrained_encoder = AutoModel.from_pretrained(
        MODEL_ID, revision=MODEL_SHA, token=HF_TOKEN,
        add_pooling_layer=False, attn_implementation='eager',
    )
    load_result = fresh_model.encoder.load_state_dict(
        pretrained_encoder.state_dict(), strict=True
    )
    if load_result.missing_keys or load_result.unexpected_keys:
        raise RuntimeError(f'Encoder copy mismatch: {load_result}')
    del pretrained_encoder
    parameter_counts = {
        'total': sum(p.numel() for p in fresh_model.parameters()),
        'trainable': sum(p.numel() for p in fresh_model.parameters() if p.requires_grad),
    }
    return fresh_model, parameter_counts

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    DataCollatorWithPadding, Trainer, TrainingArguments,
)

def metrics_from_logits(labels, logits):
    labels = np.asarray(labels)
    predicted = np.asarray(logits).argmax(axis=-1)
    _, _, f1, support = precision_recall_fscore_support(
        labels, predicted, labels=[0, 1, 2], zero_division=0
    )
    return {
        'accuracy': float(accuracy_score(labels, predicted)),
        'macro_f1': float(f1.mean()),
        'weighted_f1': float(np.average(f1, weights=support)),
    }
def compute_metrics(prediction):
    return metrics_from_logits(prediction.label_ids, prediction.predictions)
def fit_macro_f1_biases(logits, labels, radius=1.2, step=0.05):
    logits, labels = np.asarray(logits), np.asarray(labels)
    raw_score = metrics_from_logits(labels, logits)['macro_f1']
    best = (raw_score, 0.0, 0.0, 0.0)
    best_biases = np.zeros(3, dtype=np.float64)
    candidates = np.arange(-radius, radius + step / 2, step)
    for offensive_bias in candidates:
        for hate_bias in candidates:
            biases = np.array([0.0, offensive_bias, hate_bias])
            score = metrics_from_logits(labels, logits + biases)['macro_f1']
            key = (score, -float(biases @ biases), -abs(offensive_bias), -abs(hate_bias))
            if key > best:
                best, best_biases = key, biases
    return best_biases, raw_score, best[0]
def optimizer_group(model, is_encoder, use_decay, learning_rate):
    return {
        'params': [parameter for name, parameter in model.named_parameters()
                   if name.startswith('encoder.') == is_encoder
                   and (parameter.ndim > 1) == use_decay],
        'lr': learning_rate,
        'weight_decay': WEIGHT_DECAY if use_decay else 0.0,
    }
collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)
def train_and_validate(variant):
    set_seed(SEED)  # matched initialization/shuffle seed for each isolated run
    run_dir = ROOT_RUN_DIR / variant
    run_dir.mkdir(parents=True, exist_ok=False)
    model, parameters = build_fresh_model(variant)
    print({
        'phase': 'train', 'model_variant': variant,
        'architecture': ARCHITECTURE_NAMES[variant], 'parameters': parameters,
    })
    args = TrainingArguments(
        output_dir=str(run_dir / 'checkpoints'),
        num_train_epochs=EPOCHS, learning_rate=ENCODER_LR,
        per_device_train_batch_size=MICRO_BATCH, per_device_eval_batch_size=16,
        gradient_accumulation_steps=GRAD_ACCUMULATION,
        optim='adamw_torch', adam_epsilon=1e-8, weight_decay=WEIGHT_DECAY,
        lr_scheduler_type='linear', warmup_ratio=0.06, max_grad_norm=1.0,
        eval_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True, metric_for_best_model='macro_f1',
        greater_is_better=True, save_total_limit=2, fp16=True,
        seed=SEED, data_seed=SEED, report_to='none', logging_steps=50,
    )
    optimizer = torch.optim.AdamW([
        optimizer_group(model, True, True, ENCODER_LR),
        optimizer_group(model, True, False, ENCODER_LR),
        optimizer_group(model, False, True, HEAD_LR),
        optimizer_group(model, False, False, HEAD_LR),
    ], betas=(0.9, 0.999), eps=1e-8)
    trainer = Trainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=validation_ds,
        processing_class=tokenizer, data_collator=collator,
        compute_metrics=compute_metrics, optimizers=(optimizer, None),
    )
    trainer.train()
    trainer.save_model(run_dir / 'best_model')
    tokenizer.save_pretrained(run_dir / 'best_model')
    validation_output = trainer.predict(validation_ds)
    class_biases, _, calibrated_validation_macro_f1 = fit_macro_f1_biases(
        validation_output.predictions, validation_output.label_ids
    )
    validation_raw_metrics = compute_metrics(validation_output)
    print({
        'phase': 'validation', 'model_variant': variant,
        'best_checkpoint': trainer.state.best_model_checkpoint,
        **validation_raw_metrics,
        'class_biases_validation_only': class_biases.tolist(),
        'calibrated_validation_macro_f1': calibrated_validation_macro_f1,
    })
    return {
        'variant': variant, 'run_dir': run_dir, 'model': model,
        'trainer': trainer, 'parameters': parameters,
        'class_biases': class_biases,
        'validation_raw_metrics': validation_raw_metrics,
        'validation_calibrated_macro_f1': calibrated_validation_macro_f1,
    }

In [ ]:
import gc
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

test_ds = make_dataset(frames['test'])
def test_and_save(context):
    variant = context['variant']
    run_dir = context['run_dir']
    trainer = context['trainer']
    class_biases = context['class_biases']
    print({'phase': 'test', 'model_variant': variant})
    output = trainer.predict(test_ds)
    truth = np.asarray(output.label_ids)
    raw_logits = np.asarray(output.predictions)
    calibrated_logits = raw_logits + class_biases
    raw_predicted = raw_logits.argmax(axis=-1)
    predicted = calibrated_logits.argmax(axis=-1)
    raw_metrics = metrics_from_logits(truth, raw_logits)
    metrics = metrics_from_logits(truth, calibrated_logits)
    report = classification_report(
        truth, predicted, labels=[0, 1, 2],
        target_names=['CLEAN', 'OFFENSIVE', 'HATE'],
        zero_division=0, output_dict=True,
    )
    matrix = confusion_matrix(truth, predicted, labels=[0, 1, 2])
    print(json.dumps({
        'model_variant': variant, 'split': 'test',
        'class_biases_fit_on': 'validation',
        'class_biases': class_biases.tolist(),
        'raw_metrics': raw_metrics, 'calibrated_metrics': metrics,
    }, indent=2))
    display(pd.DataFrame(report).T)
    fig, ax = plt.subplots(figsize=(5, 4))
    image = ax.imshow(matrix, cmap='Purples')
    for row in range(3):
        for column in range(3):
            ax.text(column, row, int(matrix[row, column]), ha='center', va='center')
    ax.set_xticks(range(3), ['CLEAN', 'OFFENSIVE', 'HATE'])
    ax.set_yticks(range(3), ['CLEAN', 'OFFENSIVE', 'HATE'])
    ax.set(xlabel='Predicted', ylabel='True', title=f'{variant.upper()} v2 — test')
    fig.colorbar(image, ax=ax); plt.tight_layout(); plt.show()
    raw_probabilities = torch.softmax(torch.tensor(raw_logits), dim=-1).numpy()
    probabilities = torch.softmax(torch.tensor(calibrated_logits), dim=-1).numpy()
    pd.DataFrame({
        'y_true': truth, 'y_pred_raw': raw_predicted, 'y_pred': predicted,
        'p_clean_raw': raw_probabilities[:, 0],
        'p_offensive_raw': raw_probabilities[:, 1],
        'p_hate_raw': raw_probabilities[:, 2],
        'p_clean': probabilities[:, 0],
        'p_offensive': probabilities[:, 1],
        'p_hate': probabilities[:, 2],
    }).to_parquet(run_dir / 'predictions_test.parquet', index=False)
    result = {
        'experiment': f'PROPOSED_{variant.upper()}_V2',
        'model_variant': variant, 'architecture': ARCHITECTURE_NAMES[variant],
        'model_revision': MODEL_SHA,
        'dataset_revision': manifest['dataset_revision'],
        'seed': SEED, 'epochs': EPOCHS, 'effective_batch': 40,
        'cnn_kernels': list(CNN_KERNELS), 'cnn_channels': CNN_CHANNELS,
        'fusion_dim': FUSION_DIM, 'head_dropout': HEAD_DROPOUT,
        'cnn_pooling': 'masked_max_plus_mean',
        'residual_gate': variant == 'gated',
        'loss': 'class_balanced_focal', 'focal_gamma': FOCAL_GAMMA,
        'train_class_counts': train_counts.tolist(),
        'class_weights': class_weights.tolist(),
        'encoder_learning_rate': ENCODER_LR, 'head_learning_rate': HEAD_LR,
        'parameters': context['parameters'], 'evaluation_split': 'test',
        'best_checkpoint': trainer.state.best_model_checkpoint,
        'validation_raw_metrics': context['validation_raw_metrics'],
        'validation_calibrated_macro_f1': context['validation_calibrated_macro_f1'],
        'class_biases': class_biases.tolist(), 'bias_fit_split': 'validation',
        'raw_metrics': raw_metrics, 'metrics': metrics, 'run_dir': str(run_dir),
    }
    (run_dir / 'result.json').write_text(
        json.dumps(result, indent=2), encoding='utf-8'
    )
    return result

all_results = []
for variant in MODEL_VARIANTS:
    context = train_and_validate(variant)
    all_results.append(test_and_save(context))
    trainer_to_release = context.pop('trainer')
    model_to_release = context.pop('model')
    del trainer_to_release, model_to_release, context
    gc.collect()
    torch.cuda.empty_cache()

comparison = pd.DataFrame([{
    'model': item['model_variant'],
    'val_macro_f1_raw': item['validation_raw_metrics']['macro_f1'],
    'val_macro_f1_calibrated': item['validation_calibrated_macro_f1'],
    'test_macro_f1_raw': item['raw_metrics']['macro_f1'],
    'test_macro_f1_calibrated': item['metrics']['macro_f1'],
    'test_accuracy_raw': item['raw_metrics']['accuracy'],
    'test_weighted_f1_raw': item['raw_metrics']['weighted_f1'],
} for item in all_results])
display(comparison)
comparison.to_csv(ROOT_RUN_DIR / 'comparison.csv', index=False)
(ROOT_RUN_DIR / 'all_results.json').write_text(
    json.dumps(all_results, indent=2), encoding='utf-8'
)
print(f'SUCCESS: both models completed under {ROOT_RUN_DIR}')
print('Use Save Version to preserve both models and their results.')